In [1]:
%load_ext autoreload
%autoreload 2

In [16]:
OPENROUTER_API_KEY = "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"

In [35]:
# https://github.com/tommasoc80/EventStoryLine?utm_source=chatgpt.com

## EventStoryLine Dataset (Demo)

### Overview

The **EventStoryLine** dataset provides **annotated news articles** for studying **causal** and **temporal** event relations.
Each document in the corpus is represented by **three XML files** — one for the raw text, one for the annotated events, and one for the event–event relations.
The dataset is used to train and evaluate systems that reconstruct **narrative structures** or **cause–effect chains** from text.

Repository: [github.com/tommasoc80/EventStoryLine](https://github.com/tommasoc80/EventStoryLine)

---

### Folder Structure

```
EventStoryLine/
│
├── evaluation_format/
│   ├── test_corpus/
│   │   ├── 1/
│   │   │   ├── 1_10ecbplus.xml
│   │   │   ├── event_mentions_extended/
│   │   │   │   └── 1_10ecbplus.xml
│   │   │   ├── relations/
│   │   │   │   └── 1_10ecbplus.xml
│   │   │   └── coreferences/
│   │   │       └── 1_10ecbplus.xml
│   │   └── ...
│   └── dev_corpus/
└── scripts/
```

Each document (e.g., `1_10ecbplus.xml`) has:

* **Text file** → raw article text
* **Event file** → list of event mentions with unique IDs
* **Relation file** → causal or temporal links between events

---

### Example

#### 📰 Text (`text`) - INPUT

```xml
<text>
  Lindsay Lohan checks into Betty Ford Center May 03, 2013.
  Her latest stay comes after she rear-ended a truck with her Porsche on June 8.
</text>
```

#### 🔖 Event Mentions (`event_mentions_extended`) - OUTPUT

```xml
<event_mention id="6" text="checks into Betty Ford Center"/>
<event_mention id="38" text="rear-ended a truck"/>
```

#### 🔗 Relations (`relations`) - OUTPUT

```xml
<relation source="38" target="6" type="FALLING_ACTION"/>
```

#### 🧾 Real Expeted Output Format

```csv
doc_id,source_event_id,target_event_id,relation_type,source_file
1_10ecbplus.xml,38,6,FALLING_ACTION,1_10ecbplus.xml.xml
```

---

### Purpose

* Extract **event–event relations** (causal or temporal) from text.
* Build **event graphs** or **storylines** to represent narrative flow.
* Evaluate systems on **Precision**, **Recall**, and **F1** using annotated references.

### 🧩 EventStoryLine (ESC) — References and Usage

| Year     | Reference                                                                                                            | Description / Usage                                                                                  | Link                                                                                      |
| -------- | -------------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------- |
| **2017** | Caselli, T., & Vossen, P. *The Event StoryLine Corpus: A New Benchmark for Causal and Temporal Relation Extraction.* | 📘 **Original ESC dataset paper** introducing the benchmark for storyline and event causality tasks. | [ACL Anthology (W17-2711)](https://aclanthology.org/W17-2711.pdf)                         |
| **2024** | Liu, Z. et al. *Identifying while Learning for Document Event Causality.*                                            | Uses **EventStoryLine v0.9** as a benchmark for document-level event causality extraction.           | [arXiv:2405.20608v1](https://arxiv.org/html/2405.20608v1)                                 |
| **2024** | Xie, J. et al. *Semantic Aware Enhanced Event Causality Identification.*                                             | Evaluates semantic-enhanced causality extraction on the **ESC dataset**.                             | [PubMed Central](https://pmc.ncbi.nlm.nih.gov/articles/PMC11686159/)                      |
| **2023** | Liu, W. et al. *Narrative Graph: Telling Evolving Stories Based on Event-centric Knowledge Graph.*                   | Cites ESC as an early dataset for storyline reasoning and event graphs.                              | [SpringerLink](https://link.springer.com/article/10.1007/s11518-023-5561-0)               |
| **2022** | Gao, S. et al. *Document-level Causal Relation Extraction via Structured Prediction.*                                | Benchmarks against EventStoryLine for evaluating document-level event causal links.                  | [ACL Anthology (2022.findings-acl.86)](https://aclanthology.org/2022.findings-acl.86.pdf) |

---

### 🧠 Notes

* The dataset originates from the **NewsReader** project (EU FP7), focusing on cross-document storyline and causality extraction.
* **GitHub:** [tommasoc80/EventStoryLine](https://github.com/tommasoc80/EventStoryLine) hosts the original annotation scripts, corpus, and evaluation tools.
* Common usage: evaluation of **causal**, **temporal**, and **narrative event relations** in NLP pipelines.



In [3]:
# ESL (EventStoryLine) — CAT-XML + evaluation_format parser → CSVs + per-doc gold graphs
# Works with files like the ones you posted:
#   - raw_repo/annotated_data/V1.0/1_1ecbplus.xml   (CAT-XML with Markables + PLOT_LINK)
#   - raw_repo/evaluation_format/test_corpus/v1.0/event_mentions_extended/1/1_11ecbplus.xml (TSV-like links by token ids)

import os, re, json, pathlib
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Optional, Set

import pandas as pd
from lxml import etree

BASE_DIR = pathlib.Path(".").resolve()
DATA_DIR = BASE_DIR / "data" / "EventStoryLine"
RAW_DIR  = DATA_DIR / "raw_repo"
GT_DIR   = DATA_DIR / "ground_truth"
GT_DIR.mkdir(parents=True, exist_ok=True)

def clean_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()

# -----------------------------
# 1) Parse CAT-XML (gold)
# -----------------------------
docs_rows, events_rows, causal_rows = [], [], []
doc_token_text: Dict[str, Dict[str, str]] = {}      # doc_id -> t_id -> token text
doc_event_tokens: Dict[str, Dict[str, Set[str]]] = {}  # doc_id -> event_id -> set(token_ids)
doc_event_labels: Dict[str, Dict[str, str]] = {}     # doc_id -> event_id -> tag (e.g., ACTION_OCCURRENCE)
doc_id_by_filename: Dict[str, str] = {}

def parse_cat_xml(path: pathlib.Path):
    tree = etree.parse(str(path))
    root = tree.getroot()
    doc_id = root.get("doc_name") or path.stem
    doc_id_by_filename[path.name] = doc_id
    topic  = path.parent.name

    # Tokens
    token_map = {}
    ordered_tokens = []
    for tok in root.iterfind(".//token"):
        tid = tok.get("t_id") or tok.get("id")
        if not tid: 
            continue
        txt = tok.text or ""
        token_map[tid] = txt
        ordered_tokens.append((tid, txt))
    doc_token_text[doc_id] = token_map
    full_text = clean_ws(" ".join(txt for _, txt in ordered_tokens))

    # Markables → events (use ACTION_* as events; you can add more tag names if desired)
    event_tags = [t for t in {e.tag for e in root.iterfind(".//Markables/*")} if t.upper().startswith("ACTION_")]
    event_tokens: Dict[str, Set[str]] = {}
    event_labels: Dict[str, str] = {}

    def anchors_to_tokens(m) -> List[str]:
        return [a.get("t_id") for a in m.iterfind(".//token_anchor") if a.get("t_id")]

    for tag in event_tags:
        for ev in root.iterfind(f".//Markables/{tag}"):
            eid = ev.get("m_id") or ev.get("id") or ev.get("eid")
            if not eid:
                continue
            toks = anchors_to_tokens(ev)
            if not toks:
                # fall back to TAG_DESCRIPTOR if no anchors (rare late-Markables in your sample)
                span_text = ev.get("TAG_DESCRIPTOR") or ""
            else:
                span_text = clean_ws(" ".join(token_map.get(t,"") for t in toks))
            event_tokens[eid] = set(toks)
            event_labels[eid] = tag
            events_rows.append({
                "doc_id": doc_id,
                "event_id": eid,
                "event_tag": tag,
                "span_text": span_text,
                "token_ids": " ".join(toks),
                "source_file": path.name
            })

    # Relations → gold causal edges from PLOT_LINK (and CAUSAL_RELATION if present)
    def get_id_from_child(elem, child):
        node = elem.find(child)
        return node.get("m_id") if node is not None and node.get("m_id") else None

    edges = []
    # PLOT_LINK are the causal-ish links in ESL
    for rel in root.iterfind(".//Relations/PLOT_LINK"):
        rtype = (rel.get("relType") or "").upper()  # PRECONDITION, FALLING_ACTION, …
        src = get_id_from_child(rel, "source")
        tgt = get_id_from_child(rel, "target")
        if not (src and tgt): 
            continue
        edges.append((src, tgt, rtype))
        causal_rows.append({
            "doc_id": doc_id,
            "source_event_id": src,
            "target_event_id": tgt,
            "relation_type": rtype,
            "source_file": path.name
        })

    # Some releases also have CAUSAL_RELATION
    for rel in root.iterfind(".//Relations/CAUSAL_RELATION"):
        rtype = (rel.get("relType") or "CAUSE").upper()
        src = rel.get("source") or rel.get("from")
        tgt = rel.get("target") or rel.get("to")
        if not (src and tgt):
            continue
        edges.append((src, tgt, rtype))
        causal_rows.append({
            "doc_id": doc_id,
            "source_event_id": src,
            "target_event_id": tgt,
            "relation_type": rtype,
            "source_file": path.name
        })

    docs_rows.append({
        "doc_id": doc_id,
        "topic": topic,
        "doc_filename": path.name,
        "text": full_text
    })

    doc_event_tokens[doc_id] = event_tokens
    doc_event_labels[doc_id] = event_labels

# Walk CAT-XMLs
xml_files = list((RAW_DIR / "annotated_data").rglob("*.xml"))
if not xml_files:
    raise RuntimeError("No CAT-XML files found under raw_repo/annotated_data/.")
for xf in xml_files:
    parse_cat_xml(xf)

# -----------------------------
# 2) OPTIONAL: align evaluation_format token-based links to events
#    (If these exist; otherwise you already have gold from CAT-XML)
# -----------------------------
def parse_eval_line(line: str) -> Optional[Tuple[List[str], List[str], str]]:
    line = line.strip()
    if not line or line.startswith("#"):
        return None
    # lines may be "156_157\t163\tPRECONDITION" OR "34 57\t64 130 100 38_39\tPRECONDITION"
    parts = re.split(r"[\t]", line)
    if len(parts) < 3:
        return None
    lhs, rhs, rtype = parts[0].strip(), parts[1].strip(), parts[2].strip()
    def parse_side(side: str) -> List[str]:
        # split on spaces, then split pieces containing '_' into multiple token ids
        toks = []
        for chunk in side.split():
            toks.extend(chunk.split("_"))
        # keep only digits
        toks = [t for t in toks if t.isdigit()]
        return toks
    return parse_side(lhs), parse_side(rhs), rtype

def best_event_for_tokens(doc_id: str, token_ids: List[str]) -> Optional[str]:
    """Pick the event whose anchor set has the largest overlap with the given token ids."""
    if not token_ids: 
        return None
    ev_map = doc_event_tokens.get(doc_id, {})
    if not ev_map:
        return None
    tgt = set(token_ids)
    best, best_overlap = None, 0
    for eid, anchors in ev_map.items():
        ov = len(tgt & anchors)
        if ov > best_overlap:
            best, best_overlap = eid, ov
    return best

# Find evaluation files (your example path)
eval_files = list((RAW_DIR / "evaluation_format").rglob("*.xml")) + \
             list((RAW_DIR / "evaluation_format").rglob("*.tab")) + \
             list((RAW_DIR / "evaluation_format").rglob("*.tsv")) + \
             list((RAW_DIR / "evaluation_format").rglob("*.txt"))

aligned_rows = []
for ef in eval_files:
    # heuristic: if file doesn't start with '<', treat as TSV/space text
    try:
        head = ef.read_text(encoding="utf-8", errors="ignore")[:100].lstrip()
    except Exception:
        continue
    if head.startswith("<"):
        # (some eval formats are XML; skip here since we already have CAT gold)
        continue

    # deduce doc_id from filename pattern: e.g., "1_11ecbplus.xml" aligns with CAT "1_1ecbplus.xml"
    fname = ef.name
    # try to find a CAT doc that matches the prefix before first '.'
    doc_guess = fname.split(".")[0]
    # soften the guess: if no direct match, try ignoring an extra '1' after the underscore (11ecbplus vs 1ecbplus)
    candidates = {d["doc_id"]: d for d in docs_rows}
    doc_id = None
    if doc_guess in candidates:
        doc_id = doc_guess
    else:
        # try to map "1_11ecbplus" -> "1_1ecbplus"
        m = re.match(r"(\d+)_1?1(.*)$", doc_guess)
        if m:
            guess2 = f"{m.group(1)}_1{m.group(2)}"
            if guess2 in candidates:
                doc_id = guess2
    if not doc_id:
        # last resort: skip this eval file
        continue

    lines = ef.read_text(encoding="utf-8", errors="ignore").splitlines()
    for ln in lines:
        parsed = parse_eval_line(ln)
        if not parsed:
            continue
        lhs_tokens, rhs_tokens, rtype = parsed
        src_e = best_event_for_tokens(doc_id, lhs_tokens)
        tgt_e = best_event_for_tokens(doc_id, rhs_tokens)
        if src_e and tgt_e:
            aligned_rows.append({
                "doc_id": doc_id,
                "source_event_id": src_e,
                "target_event_id": tgt_e,
                "relation_type": rtype,
                "source_file": ef.as_posix(),
                "alignment": "evaluation_format→events"
            })

# Merge aligned eval links with CAT gold (keep all; drop exact dups)
extra_df = pd.DataFrame(aligned_rows)
cat_df   = pd.DataFrame(causal_rows)
if not extra_df.empty:
    merged = pd.concat([cat_df, extra_df], ignore_index=True)
    merged.drop_duplicates(subset=["doc_id","source_event_id","target_event_id","relation_type"], inplace=True)
    causal_rows = merged.to_dict("records")

# -----------------------------
# 3) Save CSVs + per-doc graphs
# -----------------------------
docs_df   = pd.DataFrame(docs_rows).drop_duplicates(subset=["doc_id"])
events_df = pd.DataFrame(events_rows)
causal_df = pd.DataFrame(causal_rows)

docs_df.to_csv(DATA_DIR / "esl_documents.csv", index=False, encoding="utf-8")
events_df.to_csv(DATA_DIR / "esl_events.csv", index=False, encoding="utf-8")
causal_df.to_csv(DATA_DIR / "esl_causal_links.csv", index=False, encoding="utf-8")

# Write per-doc JSON graphs
for doc_id in docs_df["doc_id"].tolist():
    nodes = []
    evmap = events_df[events_df.doc_id==doc_id]
    for _, r in evmap.iterrows():
        nodes.append({"id": r.event_id, "label": r.event_tag, "span_text": r.span_text, "token_ids": r.token_ids})
    edges = []
    relmap = causal_df[causal_df.doc_id==doc_id]
    for _, r in relmap.iterrows():
        edges.append({"source": r.source_event_id, "target": r.target_event_id, "relation": r.relation_type})
    out = {"doc_id": doc_id, "nodes": nodes, "edges": edges}
    (GT_DIR / f"{doc_id}.json").write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")

print("✔ Saved:", (DATA_DIR / "esl_documents.csv").as_posix())
print("✔ Saved:", (DATA_DIR / "esl_events.csv").as_posix())
print("✔ Saved:", (DATA_DIR / "esl_causal_links.csv").as_posix())

print(f"Documents: {len(docs_df):,}")
print(f"Events:    {len(events_df):,}")
print(f"Causal edges (CAT + aligned eval): {len(causal_df):,}")

# quick peek for the doc you showed if present
peek = "1_1ecbplus.xml"
if (events_df.doc_id==peek).any():
    print("\nSample events for", peek)
    display(events_df[events_df.doc_id==peek].head(10))
    print("\nSample causal links for", peek)
    display(causal_df[causal_df.doc_id==peek].head(10))

✔ Saved: C:/Users/henri/Documents/git/post-doc/ragtree/tests/data/EventStoryLine/esl_documents.csv
✔ Saved: C:/Users/henri/Documents/git/post-doc/ragtree/tests/data/EventStoryLine/esl_events.csv
✔ Saved: C:/Users/henri/Documents/git/post-doc/ragtree/tests/data/EventStoryLine/esl_causal_links.csv
Documents: 258
Events:    14,022
Causal edges (CAT + aligned eval): 4,556

Sample events for 1_1ecbplus.xml


,doc_id,event_id,event_tag,span_text,token_ids,source_file
294,1_1ecbplus.xml,5,ACTION_OCCURRENCE,treatment,138,1_1ecbplus.xml.xml
295,1_1ecbplus.xml,6,ACTION_OCCURRENCE,treatment,153,1_1ecbplus.xml.xml
296,1_1ecbplus.xml,7,ACTION_OCCURRENCE,checked into,183 184,1_1ecbplus.xml.xml
297,1_1ecbplus.xml,8,ACTION_OCCURRENCE,Leaves,34,1_1ecbplus.xml.xml
298,1_1ecbplus.xml,9,ACTION_OCCURRENCE,Checks Into,38 39,1_1ecbplus.xml.xml
299,1_1ecbplus.xml,10,ACTION_OCCURRENCE,left,57,1_1ecbplus.xml.xml
300,1_1ecbplus.xml,11,ACTION_OCCURRENCE,building,163,1_1ecbplus.xml.xml
301,1_1ecbplus.xml,12,ACTION_OCCURRENCE,moving,64,1_1ecbplus.xml.xml
302,1_1ecbplus.xml,13,ACTION_OCCURRENCE,treatment,161,1_1ecbplus.xml.xml
303,1_1ecbplus.xml,14,ACTION_OCCURRENCE,established,167,1_1ecbplus.xml.xml



Sample causal links for 1_1ecbplus.xml


,doc_id,source_event_id,target_event_id,relation_type,source_file
65,1_1ecbplus.xml,8,9,PRECONDITION,1_1ecbplus.xml.xml
66,1_1ecbplus.xml,10,12,PRECONDITION,1_1ecbplus.xml.xml
67,1_1ecbplus.xml,16,20,PRECONDITION,1_1ecbplus.xml.xml
68,1_1ecbplus.xml,30,5,FALLING_ACTION,1_1ecbplus.xml.xml
69,1_1ecbplus.xml,7,19,FALLING_ACTION,1_1ecbplus.xml.xml
70,1_1ecbplus.xml,19,15,FALLING_ACTION,1_1ecbplus.xml.xml
71,1_1ecbplus.xml,20,23,PRECONDITION,1_1ecbplus.xml.xml
72,1_1ecbplus.xml,31,13,PRECONDITION,1_1ecbplus.xml.xml
73,1_1ecbplus.xml,31,11,PRECONDITION,1_1ecbplus.xml.xml
2343,1_1ecbplus.xml,8,9,PRECONDITION,1_1ecbplus.xml.xml


In [ ]:
# %% [markdown]
# Predict ESL causal links with an OpenRouter LLM
# - Requires: OPENROUTER_API_KEY in your environment
# - Output:
#     data/EventStoryLine/predictions/pred_{doc_id}.jsonl
#     data/EventStoryLine/predictions/esl_predictions.csv

import os, json, time, re, textwrap, pathlib, uuid
from typing import List, Dict, Any, Tuple, Optional
import requests
import pandas as pd

DATA_DIR = pathlib.Path("data/EventStoryLine")
DOCS_CSV   = DATA_DIR / "esl_documents.csv"
EVENTS_CSV = DATA_DIR / "esl_events.csv"
GOLD_CSV   = DATA_DIR / "esl_causal_links.csv"
PRED_DIR   = DATA_DIR / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
#OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")  # set this in your shell or notebook env

# ------------- helpers
def _clean(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip().lower()

def _jaccard(a: str, b: str) -> float:
    sa = set(_clean(a).split())
    sb = set(_clean(b).split())
    return 0.0 if not sa or not sb else len(sa & sb) / len(sa | sb)

def _best_event_for_span(doc_events: pd.DataFrame, span_text: str, min_sim: float = 0.2) -> Optional[str]:
    """Pick the event_id with the highest token Jaccard against span_text."""
    if not span_text or doc_events.empty:
        return None
    best_id, best_sim = None, 0.0
    for _, r in doc_events.iterrows():
        sim = _jaccard(span_text, r["span_text"])
        if sim > best_sim:
            best_id, best_sim = r["event_id"], sim
    return best_id if best_sim >= min_sim else None

def _extract_json(text: str) -> Dict[str, Any]:
    """
    Try to parse JSON from the model output.
    Accepts raw JSON or fenced code blocks ```json ... ```.
    """
    if not text:
        return {}
    # fenced block
    m = re.search(r"```json\s*(\{.*?\})\s*```", text, flags=re.S|re.I)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass
    # first top-level JSON object
    try:
        return json.loads(text.strip())
    except Exception:
        pass
    # last resort: find {...} region
    m = re.search(r"(\{.*\})", text, flags=re.S)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass
    return {}

def _openrouter_call(prompt: str,
                     model: str = "openai/gpt-4o-mini",
                     temperature: float = 0.0,
                     max_tokens: int = 1500,
                     retries: int = 2,
                     sleep_s: float = 2.0) -> str:
    if not OPENROUTER_API_KEY:
        raise RuntimeError("Set OPENROUTER_API_KEY in your environment before calling the model.")
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        # optional routing metadata:
        "HTTP-Referer": "https://example.org/your-app", 
        "X-Title": "ESL causal extraction",
    }
    body = {
        "model": model,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You extract CAUSAL relations from a single document. "
                    "Return STRICT JSON only, matching the given schema. "
                    "Use verbatim substrings from the document for source_text and target_text."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
    }
    for attempt in range(retries + 1):
        try:
            resp = requests.post(OPENROUTER_URL, headers=headers, data=json.dumps(body), timeout=120)
            if resp.status_code == 200:
                data = resp.json()
                return data["choices"][0]["message"]["content"]
            else:
                err = f"OpenRouter HTTP {resp.status_code}: {resp.text[:300]}"
                if attempt < retries:
                    time.sleep(sleep_s)
                    continue
                raise RuntimeError(err)
        except Exception as e:
            if attempt < retries:
                time.sleep(sleep_s)
                continue
            raise
    return ""

def _build_prompt(doc_text: str) -> str:
    allowed = ["PRECONDITION", "FALLING_ACTION", "CAUSE", "RESULT", "ENABLE"]
    schema = {
        "type": "object",
        "properties": {
            "relations": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "source_text": {"type": "string"},
                        "target_text": {"type": "string"},
                        "relation_type": {"type": "string", "enum": allowed}
                    },
                    "required": ["source_text", "target_text", "relation_type"],
                    "additionalProperties": False
                }
            }
        },
        "required": ["relations"],
        "additionalProperties": False
    }
    return textwrap.dedent(f"""
    Extract all document-level CAUSAL relations from the text below.

    • Only include causal-like types from this set: {allowed}.
    • For source_text and target_text, copy the exact substrings from the document (short, ~1–6 tokens).
    • If none exist, return {{"relations": []}}.
    • Return STRICT JSON ONLY (no prose), matching this schema:
    {json.dumps(schema, indent=2)}

    Document:
    ---
    {doc_text}
    ---
    Output JSON:
    """).strip()

# ------------- main function
def predict_esl_relations_openrouter(n: int = 1,
                                     model: str = "openai/gpt-4o-mini",
                                     min_span_event_sim: float = 0.2) -> pd.DataFrame:
    """
    Picks the first n ESL docs (with gold edges available), calls OpenRouter to extract causal links,
    aligns predicted spans to gold events, and saves predictions.
    Returns a DataFrame with one row per predicted edge (aligned to event_ids when possible).
    """
    docs = pd.read_csv(DOCS_CSV)
    events = pd.read_csv(EVENTS_CSV)
    gold = pd.read_csv(GOLD_CSV)

    # choose docs that *have* gold edges; if none, take any docs
    docs_with_gold = gold["doc_id"].unique().tolist()
    if docs_with_gold:
        subset = docs[docs["doc_id"].isin(docs_with_gold)].head(n)
    else:
        subset = docs.head(n)

    all_pred_rows: List[Dict[str, Any]] = []

    for _, drow in subset.iterrows():
        doc_id = drow["doc_id"]
        text   = drow["text"]
        print(f"\n⏳ Predicting relations for: {doc_id}")

        prompt = _build_prompt(text)
        raw = _openrouter_call(prompt, model=model)

        data = _extract_json(raw)
        rels = data.get("relations", []) if isinstance(data, dict) else []

        # map predicted spans to nearest event ids
        evs_doc = events[events["doc_id"] == doc_id]
        mapped = []
        for r in rels:
            src_txt = r.get("source_text", "")
            tgt_txt = r.get("target_text", "")
            rtype   = (r.get("relation_type", "") or "").upper()

            src_id = _best_event_for_span(evs_doc, src_txt, min_sim=min_span_event_sim)
            tgt_id = _best_event_for_span(evs_doc, tgt_txt, min_sim=min_span_event_sim)

            mapped.append({
                "doc_id": doc_id,
                "pred_source_span": src_txt,
                "pred_target_span": tgt_txt,
                "pred_relation_type": rtype,
                "pred_source_event_id": src_id,
                "pred_target_event_id": tgt_id,
                "model": model,
                "raw_json": json.dumps(r, ensure_ascii=False)
            })

        # save per-doc JSONL
        out_jsonl = PRED_DIR / f"pred_{doc_id}.jsonl"
        with out_jsonl.open("w", encoding="utf-8") as fh:
            for r in mapped:
                fh.write(json.dumps(r, ensure_ascii=False) + "\n")

        all_pred_rows.extend(mapped)
        print(f"✅ {doc_id}: {len(mapped)} predicted edges (aligned: "
              f"{sum(1 for m in mapped if m['pred_source_event_id'] and m['pred_target_event_id'])})")

    # combined CSV
    pred_df = pd.DataFrame(all_pred_rows)
    pred_csv = PRED_DIR / "esl_predictions.csv"
    pred_df.to_csv(pred_csv, index=False, encoding="utf-8")
    print(f"\n✔ Saved predictions CSV: {pred_csv}")
    return pred_df

# Example usage:
# preds = predict_esl_relations_openrouter(n=1, model="anthropic/claude-3.5-sonnet")
# preds.head()


In [40]:
preds = predict_esl_relations_openrouter(n=10, model="openai/gpt-4o-mini")
preds.head()


⏳ Predicting relations for: 1_10ecbplus.xml
✅ 1_10ecbplus.xml: 3 predicted edges (aligned: 2)

⏳ Predicting relations for: 1_11ecbplus.xml
✅ 1_11ecbplus.xml: 2 predicted edges (aligned: 1)

⏳ Predicting relations for: 1_12ecbplus.xml
✅ 1_12ecbplus.xml: 4 predicted edges (aligned: 2)

⏳ Predicting relations for: 1_13ecbplus.xml
✅ 1_13ecbplus.xml: 3 predicted edges (aligned: 1)

⏳ Predicting relations for: 1_14ecbplus.xml
✅ 1_14ecbplus.xml: 3 predicted edges (aligned: 1)

⏳ Predicting relations for: 1_15ecbplus.xml
✅ 1_15ecbplus.xml: 3 predicted edges (aligned: 3)

⏳ Predicting relations for: 1_16ecbplus.xml
✅ 1_16ecbplus.xml: 2 predicted edges (aligned: 2)

⏳ Predicting relations for: 1_18ecbplus.xml
✅ 1_18ecbplus.xml: 0 predicted edges (aligned: 0)

⏳ Predicting relations for: 1_19ecbplus.xml
✅ 1_19ecbplus.xml: 3 predicted edges (aligned: 2)

⏳ Predicting relations for: 1_1ecbplus.xml
✅ 1_1ecbplus.xml: 2 predicted edges (aligned: 2)

✔ Saved predictions CSV: data\EventStoryLine\predic

,doc_id,pred_source_span,pred_target_span,pred_relation_type,pred_source_event_id,pred_target_event_id,model,raw_json
0,1_10ecbplus.xml,facing the prospect,checked into the Betty Ford Center,CAUSE,4.0,5.0,openai/gpt-4o-mini,"{""source_text"": ""facing the prospect"", ""target..."
1,1_10ecbplus.xml,court-mandated stay,checked into the Betty Ford Center,PRECONDITION,7.0,5.0,openai/gpt-4o-mini,"{""source_text"": ""court-mandated stay"", ""target..."
2,1_10ecbplus.xml,rear-ended a truck,checked into the Betty Ford Center,CAUSE,NaN,5.0,openai/gpt-4o-mini,"{""source_text"": ""rear-ended a truck"", ""target_..."
3,1_11ecbplus.xml,checked into rehab,dodging arrest,RESULT,4.0,NaN,openai/gpt-4o-mini,"{""source_text"": ""checked into rehab"", ""target_..."
4,1_11ecbplus.xml,hiring former lawyer Shawn Holley,help,ENABLE,5.0,9.0,openai/gpt-4o-mini,"{""source_text"": ""hiring former lawyer Shawn Ho..."


In [41]:
# Robust soft rematching that guarantees an event id if the doc has events
import re, math
import pandas as pd
from difflib import SequenceMatcher
from pathlib import Path

DATA_DIR = Path("data/EventStoryLine")
DOCS   = pd.read_csv(DATA_DIR / "esl_documents.csv")
EVENTS = pd.read_csv(DATA_DIR / "esl_events.csv")
PREDS  = pd.read_csv(DATA_DIR / "predictions" / "esl_predictions.csv")

STOP = set("the a an of to in on at for from with and or but by is are was were be been being this that it she he they them her his their its as into over under out up down".split())

def norm(s):
    s = (s or "").lower()
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def content_tokens(s):
    return [t for t in norm(s).split() if t not in STOP]

def jaccard(a,b):
    A=set(content_tokens(a)); B=set(content_tokens(b))
    return 0.0 if not A or not B else len(A&B)/len(A|B)

def difflib_sim(a,b): return SequenceMatcher(None, norm(a), norm(b)).ratio()

def containment(a,b):
    na, nb = " ".join(content_tokens(a)), " ".join(content_tokens(b))
    if not na or not nb: return 0.0
    if na in nb: return len(na)/max(1,len(nb))
    if nb in na: return len(nb)/max(1,len(na))
    return 0.0

def char_pos(text, span):
    t = norm(text); s = norm(span)
    if not s: return None
    idx = t.find(s)
    return idx if idx >= 0 else None

def soft_best_event(doc_id, span_text, neighbor_pos=None):
    """
    Always returns (event_id, score). If no events exist: (None, 0.0).
    neighbor_pos: int | None – if provided, we add a small proximity bias to this char position.
    """
    evs = EVENTS[EVENTS.doc_id==doc_id][["event_id","span_text","event_tag"]].copy()
    if evs.empty:
        return None, 0.0
    text = DOCS[DOCS.doc_id==doc_id].iloc[0]["text"]

    # scoring
    rows=[]
    for _, ev in evs.iterrows():
        sj = jaccard(span_text, ev.span_text)
        sd = difflib_sim(span_text, ev.span_text)
        sc = containment(span_text, ev.span_text)
        score = 0.45*sj + 0.45*sd + 0.10*sc

        # event-tag prior: bump true actions slightly
        if isinstance(ev.event_tag, str) and ev.event_tag.startswith("ACTION_"):
            score += 0.03

        # proximity bias if we know the other side's char position
        if neighbor_pos is not None:
            pos_ev = char_pos(text, ev.span_text)
            if pos_ev is not None:
                dist = abs(neighbor_pos - pos_ev)
                score -= 0.02*math.log1p(dist)

        rows.append((score, ev.event_id))
    rows.sort(reverse=True)
    return rows[0][1], rows[0][0]

def rematch_predictions_soft_guaranteed():
    rows=[]
    for doc_id, pred_df in PREDS.groupby("doc_id"):
        text = DOCS[DOCS.doc_id==doc_id].iloc[0]["text"]
        for _, r in pred_df.iterrows():
            src_span = r.get("pred_source_span","") or ""
            tgt_span = r.get("pred_target_span","") or ""

            # positions (for proximity bias)
            src_pos = char_pos(text, src_span)
            tgt_pos = char_pos(text, tgt_span)

            # if an id is already present, keep it, otherwise find best with neighbor context
            src_id = r.get("pred_source_event_id")
            if pd.isna(src_id) or not isinstance(src_id, str):
                src_id, src_score = soft_best_event(doc_id, src_span, neighbor_pos=tgt_pos)
            else:
                src_score = 1.0

            tgt_id = r.get("pred_target_event_id")
            if pd.isna(tgt_id) or not isinstance(tgt_id, str):
                tgt_id, tgt_score = soft_best_event(doc_id, tgt_span, neighbor_pos=src_pos)
            else:
                tgt_score = 1.0

            out = dict(r)
            out["pred_source_event_id_soft"] = src_id
            out["pred_target_event_id_soft"] = tgt_id
            out["src_match_score"] = src_score
            out["tgt_match_score"] = tgt_score
            rows.append(out)

    out_df = pd.DataFrame(rows)
    out_path = DATA_DIR / "predictions" / "esl_predictions_soft.csv"
    out_df.to_csv(out_path, index=False, encoding="utf-8")
    print("Saved:", out_path, " | rows:", len(out_df))
    # sanity: report NaNs (should be zero unless a doc truly has no events)
    print("NaNs (source):", out_df["pred_source_event_id_soft"].isna().sum(),
          "NaNs (target):", out_df["pred_target_event_id_soft"].isna().sum())
    return out_df

soft_preds2 = rematch_predictions_soft_guaranteed()
soft_preds2.head()


Saved: data\EventStoryLine\predictions\esl_predictions_soft.csv  | rows: 25
NaNs (source): 0 NaNs (target): 0


,doc_id,pred_source_span,pred_target_span,pred_relation_type,pred_source_event_id,pred_target_event_id,model,raw_json,pred_source_event_id_soft,pred_target_event_id_soft,src_match_score,tgt_match_score
0,1_10ecbplus.xml,facing the prospect,checked into the Betty Ford Center,CAUSE,4.0,5.0,openai/gpt-4o-mini,"{""source_text"": ""facing the prospect"", ""target...",4,5,0.424124,0.318407
1,1_10ecbplus.xml,court-mandated stay,checked into the Betty Ford Center,PRECONDITION,7.0,5.0,openai/gpt-4o-mini,"{""source_text"": ""court-mandated stay"", ""target...",44,5,0.272892,0.325503
2,1_10ecbplus.xml,rear-ended a truck,checked into the Betty Ford Center,CAUSE,NaN,5.0,openai/gpt-4o-mini,"{""source_text"": ""rear-ended a truck"", ""target_...",8,65,0.600809,0.295574
3,1_11ecbplus.xml,checked into rehab,dodging arrest,RESULT,4.0,NaN,openai/gpt-4o-mini,"{""source_text"": ""checked into rehab"", ""target_...",6,3,0.556008,0.201429
4,1_11ecbplus.xml,hiring former lawyer Shawn Holley,help,ENABLE,5.0,9.0,openai/gpt-4o-mini,"{""source_text"": ""hiring former lawyer Shawn Ho...",5,13,0.203372,0.956729


In [43]:
# Robust ESL validator (handles NaN/float labels safely)

import json, pathlib
from typing import Dict, Any, Set, Tuple
import pandas as pd

try:
    import networkx as nx
except Exception:
    nx = None

DATA_DIR = pathlib.Path("data/EventStoryLine")
GOLD_PATH = DATA_DIR / "esl_causal_links.csv"
PRED_FINAL_PATH = DATA_DIR / "predictions" / "esl_predictions_final.csv"
PRED_SOFT_PATH  = DATA_DIR / "predictions" / "esl_predictions_soft.csv"

CAUSAL_SET = {"PRECONDITION","CAUSE","RESULT","ENABLE","FALLING_ACTION","CAUSAL_RELATION"}

def _safe_upper(x) -> str:
    if pd.isna(x):
        return ""
    s = str(x)
    return "" if s.lower() == "nan" else s.upper()

def _norm_lbl(lbl, strategy: str) -> str:
    L = _safe_upper(lbl)
    if strategy == "binary":
        return "CAUSAL" if L in CAUSAL_SET else L
    return L  # 'strict' or 'ignore' (ignore is handled later)

def _edge_set(df: pd.DataFrame, label_strategy: str, use_label: bool) -> Set[Tuple[str,str,str]]:
    out = set()
    for s,t,r in df[["source_event_id","target_event_id","relation_type"]].itertuples(index=False, name=None):
        s = str(s); t = str(t)
        lab = _norm_lbl(r, label_strategy) if use_label else ""
        out.add((s,t,lab))
    return out

def _choose_predictions_df() -> pd.DataFrame:
    if PRED_FINAL_PATH.exists():
        df = pd.read_csv(PRED_FINAL_PATH)
        df = df.rename(columns={
            "source_event_id_final": "source_event_id",
            "target_event_id_final": "target_event_id",
            "pred_relation_type": "relation_type"
        })
    elif PRED_SOFT_PATH.exists():
        df = pd.read_csv(PRED_SOFT_PATH)
        src_col = "pred_source_event_id_soft" if "pred_source_event_id_soft" in df.columns else "pred_source_event_id"
        tgt_col = "pred_target_event_id_soft" if "pred_target_event_id_soft" in df.columns else "pred_target_event_id"
        df = df.rename(columns={src_col:"source_event_id", tgt_col:"target_event_id", "pred_relation_type":"relation_type"})
    else:
        raise FileNotFoundError("No prediction file found.")

    # ensure required cols and types
    needed = {"doc_id","source_event_id","target_event_id","relation_type"}
    missing = needed - set(df.columns)
    if missing:
        raise RuntimeError(f"Predictions missing columns: {missing}")
    df = df[list(needed)].copy()
    for c in ("doc_id","source_event_id","target_event_id","relation_type"):
        df[c] = df[c].astype(object).where(pd.notna(df[c]), "")
    return df

def validate_esl(n: int = 1, label_strategy: str = "strict", use_transitive: bool = False) -> Dict[str, Any]:
    gold = pd.read_csv(GOLD_PATH)
    # sanitize gold
    for c in ("doc_id","source_event_id","target_event_id","relation_type"):
        if c in gold.columns:
            gold[c] = gold[c].astype(object).where(pd.notna(gold[c]), "")
        else:
            raise RuntimeError(f"Gold missing column: {c}")
    pred = _choose_predictions_df()

    # select docs
    doc_ids = list(dict.fromkeys(pred["doc_id"].tolist()))[:n]
    if not doc_ids:
        raise RuntimeError("No predictions found.")
    gold = gold[gold["doc_id"].isin(doc_ids)].copy()
    pred = pred[pred["doc_id"].isin(doc_ids)].copy()

    per_doc_rows = []
    micro_TP = micro_FP = micro_FN = 0
    use_label = (label_strategy == "strict")

    for d in doc_ids:
        Gd = gold[gold["doc_id"]==d][["source_event_id","target_event_id","relation_type"]].copy()
        Pd = pred[pred["doc_id"]==d][["source_event_id","target_event_id","relation_type"]].copy()

        # normalize id columns to string
        for c in ("source_event_id","target_event_id"):
            Gd[c] = Gd[c].astype(str); Pd[c] = Pd[c].astype(str)

        gset = _edge_set(Gd, label_strategy, use_label)
        pset = _edge_set(Pd, label_strategy, use_label)

        TP = len(gset & pset)
        FP = len(pset - gset)
        FN = len(gset - pset)

        P  = TP/(TP+FP) if (TP+FP) else 0.0
        R  = TP/(TP+FN) if (TP+FN) else 0.0
        F1 = 2*P*R/(P+R) if (P+R) else 0.0

        micro_TP += TP; micro_FP += FP; micro_FN += FN

        per_doc_rows.append({
            "doc_id": d, "gold_edges": len(gset), "pred_edges": len(pset),
            "TP": TP, "FP": FP, "FN": FN, "precision": P, "recall": R, "f1": F1
        })

    per_doc_df = pd.DataFrame(per_doc_rows)
    macro_P = per_doc_df["precision"].mean() if not per_doc_df.empty else 0.0
    macro_R = per_doc_df["recall"].mean() if not per_doc_df.empty else 0.0
    macro_F1 = per_doc_df["f1"].mean() if not per_doc_df.empty else 0.0
    micro_P = micro_TP/(micro_TP+micro_FP) if (micro_TP+micro_FP) else 0.0
    micro_R = micro_TP/(micro_TP+micro_FN) if (micro_TP+micro_FN) else 0.0
    micro_F1 = 2*micro_P*micro_R/(micro_P+micro_R) if (micro_P+micro_R) else 0.0

    # save artifacts
    edges_out = DATA_DIR / "predictions" / "esl_eval_edges.csv"
    per_doc_df.to_csv(edges_out, index=False, encoding="utf-8")

    summary = {
        "docs_evaluated": doc_ids,
        "label_strategy": label_strategy,
        "macro": {"precision": macro_P, "recall": macro_R, "f1": macro_F1},
        "micro": {"precision": micro_P, "recall": micro_R, "f1": micro_F1},
        "counts": {"TP": micro_TP, "FP": micro_FP, "FN": micro_FN}
    }
    summ_out = DATA_DIR / "predictions" / "esl_eval_summary.json"
    summ_out.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print(f"Saved per-doc edges: {edges_out}")
    print(f"Saved summary JSON : {summ_out}")
    print(f"\nMicro  P/R/F1 = {micro_P:.3f} / {micro_R:.3f} / {micro_F1:.3f}")
    print(f"Macro  P/R/F1 = {macro_P:.3f} / {macro_R:.3f} / {macro_F1:.3f}")

    return {"per_doc": per_doc_df, "summary": summary}

# Example:
res = validate_esl(n=10, label_strategy="strict", use_transitive=False)
res["per_doc"].head()


Saved per-doc edges: data\EventStoryLine\predictions\esl_eval_edges.csv
Saved summary JSON : data\EventStoryLine\predictions\esl_eval_summary.json

Micro  P/R/F1 = 0.000 / 0.000 / 0.000
Macro  P/R/F1 = 0.000 / 0.000 / 0.000


,doc_id,gold_edges,pred_edges,TP,FP,FN,precision,recall,f1
0,1_10ecbplus.xml,17,3,0,3,17,0.0,0.0,0.0
1,1_11ecbplus.xml,12,2,0,2,12,0.0,0.0,0.0
2,1_12ecbplus.xml,13,4,0,4,13,0.0,0.0,0.0
3,1_13ecbplus.xml,10,3,0,3,10,0.0,0.0,0.0
4,1_14ecbplus.xml,14,3,0,3,14,0.0,0.0,0.0


In [44]:
# %% [markdown]
# Soft ESL evaluation (partial credit for head-only / tail-only / undirected)
# Inputs:
#   data/EventStoryLine/esl_causal_links.csv
#   data/EventStoryLine/predictions/esl_predictions_final.csv (preferred)
#   or .../esl_predictions_soft.csv (fallback)
# Output:
#   prints per-doc + micro/macro + soft scores, returns dict of DataFrames

import json, pathlib
from typing import Dict, Any, Tuple, Set
import pandas as pd

DATA_DIR = pathlib.Path("data/EventStoryLine")
GOLD_PATH = DATA_DIR / "esl_causal_links.csv"
PRED_FINAL_PATH = DATA_DIR / "predictions" / "esl_predictions_final.csv"
PRED_SOFT_PATH  = DATA_DIR / "predictions" / "esl_predictions_soft.csv"

CAUSAL_SET = {"PRECONDITION","CAUSE","RESULT","ENABLE","FALLING_ACTION","CAUSAL_RELATION"}

def _safe_upper(x) -> str:
    if pd.isna(x): return ""
    s = str(x)
    return "" if s.lower() == "nan" else s.upper()

def _norm_lbl(lbl, strategy: str) -> str:
    L = _safe_upper(lbl)
    if strategy == "binary":
        return "CAUSAL" if L in CAUSAL_SET else L
    return L  # 'strict' or 'ignore' (ignore handled downstream)

def _load_gold_pred():
    gold = pd.read_csv(GOLD_PATH)
    for c in ("doc_id","source_event_id","target_event_id","relation_type"):
        gold[c] = gold[c].astype(object).where(pd.notna(gold[c]), "")
    # predictions: prefer final ids, else soft ids
    if PRED_FINAL_PATH.exists():
        pred = pd.read_csv(PRED_FINAL_PATH)
        pred = pred.rename(columns={
            "source_event_id_final": "source_event_id",
            "target_event_id_final": "target_event_id",
            "pred_relation_type": "relation_type"
        })
    else:
        pred = pd.read_csv(PRED_SOFT_PATH)
        src_col = "pred_source_event_id_soft" if "pred_source_event_id_soft" in pred.columns else "pred_source_event_id"
        tgt_col = "pred_target_event_id_soft" if "pred_target_event_id_soft" in pred.columns else "pred_target_event_id"
        pred = pred.rename(columns={src_col:"source_event_id", tgt_col:"target_event_id", "pred_relation_type":"relation_type"})
    for c in ("doc_id","source_event_id","target_event_id","relation_type"):
        pred[c] = pred[c].astype(object).where(pd.notna(pred[c]), "")
    return gold, pred

def evaluate_esl_soft(n: int = 1,
                      label_strategy: str = "strict",
                      weights: Dict[str, float] = None) -> Dict[str, Any]:
    """
    label_strategy: 'strict' | 'ignore' | 'binary'
      - 'strict'  : (src, tgt, label) must match for EXACT
      - 'ignore'  : labels ignored for EXACT/UNDIR; still reported per-type separately if needed
      - 'binary'  : collapse causal-like labels to 'CAUSAL' before comparison
    weights: partial-credit weights:
      {'exact':1.0, 'undirected':0.75, 'one_end':0.5}
    """
    if weights is None:
        weights = {"exact": 1.0, "undirected": 0.75, "one_end": 0.5}

    gold, pred = _load_gold_pred()

    # doc subset (first n with predictions)
    doc_ids = list(dict.fromkeys(pred["doc_id"].tolist()))[:n]
    gold = gold[gold["doc_id"].isin(doc_ids)].copy()
    pred = pred[pred["doc_id"].isin(doc_ids)].copy()

    use_label = (label_strategy == "strict")

    per_doc_rows = []
    micro_exact = micro_undir = micro_one = 0.0
    micro_pred_edges = 0
    micro_gold_edges = 0

    def sets_for_doc(Gd: pd.DataFrame, Pd: pd.DataFrame):
        # normalize ids -> str
        for c in ("source_event_id","target_event_id"):
            Gd[c] = Gd[c].astype(str); Pd[c] = Pd[c].astype(str)
        # label handling
        Gd["relation_type_norm"] = Gd["relation_type"].map(lambda x: _norm_lbl(x, label_strategy))
        Pd["relation_type_norm"] = Pd["relation_type"].map(lambda x: _norm_lbl(x, label_strategy))
        # exact sets (with or without label)
        G_exact = set((a,b,(t if use_label else "")) for a,b,t in Gd[["source_event_id","target_event_id","relation_type_norm"]].itertuples(index=False, name=None))
        P_exact = set((a,b,(t if use_label else "")) for a,b,t in Pd[["source_event_id","target_event_id","relation_type_norm"]].itertuples(index=False, name=None))
        # undirected (label-agnostic)
        G_undir = set(tuple(sorted((a,b))) for a,b in Gd[["source_event_id","target_event_id"]].itertuples(index=False, name=None))
        P_undir = set(tuple(sorted((a,b))) for a,b in Pd[["source_event_id","target_event_id"]].itertuples(index=False, name=None))
        # endpoint bags
        G_heads = set(Gd["source_event_id"].astype(str))
        G_tails = set(Gd["target_event_id"].astype(str))
        return G_exact, P_exact, G_undir, P_undir, G_heads, G_tails

    details_rows = []

    for d in doc_ids:
        Gd = gold[gold["doc_id"]==d][["source_event_id","target_event_id","relation_type"]].copy()
        Pd = pred[pred["doc_id"]==d][["source_event_id","target_event_id","relation_type"]].copy()

        G_exact, P_exact, G_undir, P_undir, G_heads, G_tails = sets_for_doc(Gd, Pd)

        exact_tp = len(G_exact & P_exact)
        undir_tp = len(P_undir - set(tuple(sorted((a,b))) for a,b,_ in (G_exact & P_exact))) & 0  # placeholder (we'll compute per-pred)
        one_end_tp = 0

        # compute soft credits per predicted edge
        exact_ids = set((a,b) for a,b,_ in (G_exact & P_exact))  # for excluding from partials
        undir_only = 0
        one_only   = 0

        for a,b,t in P_exact:
            micro_pred_edges += 1
            # exact?
            if (a,b,(t if use_label else "")) in G_exact:
                credit = weights["exact"]; kind="exact"
            else:
                # undirected?
                if tuple(sorted((a,b))) in G_undir:
                    credit = weights.get("undirected", 0.75); kind="undirected"
                    undir_only += 1
                else:
                    # one endpoint right?
                    head_ok = a in G_heads
                    tail_ok = b in G_tails
                    if head_ok or tail_ok:
                        credit = weights.get("one_end", 0.5); kind="one_end"
                        one_only += 1
                    else:
                        credit = 0.0; kind="wrong"

            details_rows.append({
                "doc_id": d, "pred_source": a, "pred_target": b,
                "pred_label": t, "match_kind": kind, "credit": credit
            })

        gold_edges = len(G_exact)  # count of gold edges (label-aware or not)
        micro_gold_edges += gold_edges

        # sum credits for micro
        doc_credit_exact = sum(r["credit"] for r in details_rows if r["doc_id"]==d and r["match_kind"]=="exact")
        doc_credit_undir = sum(r["credit"] for r in details_rows if r["doc_id"]==d and r["match_kind"]=="undirected")
        doc_credit_one   = sum(r["credit"] for r in details_rows if r["doc_id"]==d and r["match_kind"]=="one_end")

        micro_exact += doc_credit_exact
        micro_undir += doc_credit_undir
        micro_one   += doc_credit_one

        # hard counts for exact-only P/R/F1 (for reference)
        hard_TP = exact_tp
        hard_FP = len(P_exact) - exact_tp
        hard_FN = len(G_exact) - exact_tp
        hard_P  = hard_TP/(hard_TP+hard_FP) if (hard_TP+hard_FP) else 0.0
        hard_R  = hard_TP/(hard_TP+hard_FN) if (hard_TP+hard_FN) else 0.0
        hard_F1 = 2*hard_P*hard_R/(hard_P+hard_R) if (hard_P+hard_R) else 0.0

        per_doc_rows.append({
            "doc_id": d,
            "gold_edges": gold_edges,
            "pred_edges": len(P_exact),
            "hard_precision": hard_P, "hard_recall": hard_R, "hard_f1": hard_F1,
            "credits_exact_sum": doc_credit_exact,
            "credits_undirected_sum": doc_credit_undir,
            "credits_one_end_sum": doc_credit_one
        })

    # micro soft precision/recall (credit-based)
    total_credit = micro_exact + micro_undir + micro_one
    soft_precision = total_credit / micro_pred_edges if micro_pred_edges else 0.0
    soft_recall    = total_credit / micro_gold_edges if micro_gold_edges else 0.0
    soft_f1        = 2*soft_precision*soft_recall/(soft_precision+soft_recall) if (soft_precision+soft_recall) else 0.0

    # summarize
    per_doc_df = pd.DataFrame(per_doc_rows)
    details_df = pd.DataFrame(details_rows)

    print(f"Docs: {len(doc_ids)}  | Pred edges: {micro_pred_edges}  | Gold edges: {micro_gold_edges}")
    print(f"HARD (exact only) — macro F1: {per_doc_df['hard_f1'].mean():.3f} | "
          f"micro F1: { (lambda TP,FP,FN: (2*(TP/(TP+FP))*(TP/(TP+FN))/((TP/(TP+FP))+(TP/(TP+FN)))) if TP else 0.0)(0,0,0) }")
    print(f"SOFT (credit-based; weights={weights}) — micro P/R/F1: "
          f"{soft_precision:.3f} / {soft_recall:.3f} / {soft_f1:.3f}")
    print(f"  credit breakdown — exact: {micro_exact:.2f}, undirected: {micro_undir:.2f}, one_end: {micro_one:.2f}")

    return {
        "per_doc": per_doc_df,
        "per_pred_details": details_df,
        "soft_summary": {
            "weights": weights,
            "micro_precision": soft_precision,
            "micro_recall": soft_recall,
            "micro_f1": soft_f1,
            "credit_exact": micro_exact,
            "credit_undirected": micro_undir,
            "credit_one_end": micro_one,
            "pred_edges": micro_pred_edges,
            "gold_edges": micro_gold_edges,
            "docs": doc_ids
        }
    }

# Example:
soft_res = evaluate_esl_soft(n=10, label_strategy="binary",
                             weights={"exact":1.0, "undirected":0.75, "one_end":0.5})
soft_res["per_doc"].head()


Docs: 9  | Pred edges: 25  | Gold edges: 97
HARD (exact only) — macro F1: 0.097 | micro F1: 0.0
SOFT (credit-based; weights={'exact': 1.0, 'undirected': 0.75, 'one_end': 0.5}) — micro P/R/F1: 0.430 / 0.111 / 0.176
  credit breakdown — exact: 4.00, undirected: 0.75, one_end: 6.00


,doc_id,gold_edges,pred_edges,hard_precision,hard_recall,hard_f1,credits_exact_sum,credits_undirected_sum,credits_one_end_sum
0,1_10ecbplus.xml,17,3,0.0,0.0,0.0,0.0,0.75,0.5
1,1_11ecbplus.xml,12,2,0.0,0.0,0.0,0.0,0.00,0.5
2,1_12ecbplus.xml,13,4,0.0,0.0,0.0,0.0,0.00,1.0
3,1_13ecbplus.xml,8,3,0.0,0.0,0.0,0.0,0.00,1.0
4,1_14ecbplus.xml,14,3,0.0,0.0,0.0,0.0,0.00,1.0
